# SCB-TumorSeg · Đánh giá mô hình tuần 2

Đánh giá M1 (3D ResUNet) và M2 (3D Attention U-Net) trên split test MAMA-MIA. Mỗi checkpoint cho hai kết quả P0 và P1 trên cùng bản đồ xác suất. Notebook xuất điểm theo ca và bảng tổng hợp theo cấu hình.

## 1. Môi trường và cấu hình

Mở notebook trong repo. Kernel cần PyTorch phù hợp với thiết bị và các gói trong `requirements.txt`. Trên Colab, đặt thư mục làm việc trong repo trước khi chạy.

Dữ liệu đầu vào gồm các NPZ đã qua Steps 1–3 và `splits_metadata.csv` của nhóm. Cấu hình bên dưới chạy một ca test với một checkpoint. Đặt `MAX_CASES = None` để đánh giá toàn bộ split test. Với 12 checkpoint M1/M2, bảng tổng hợp có 24 dòng.

In [ ]:
import sys
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from monai.inferers import sliding_window_inference
from monai.metrics import compute_hausdorff_distance
from scipy import ndimage

ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "configs/default_config.yaml").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Mở notebook từ bên trong thư mục repo SCB-TumorSeg.")
sys.path.insert(0, str(ROOT))

from src.datasets import MAMA_MIA_Dataset
from src.models import build_model

NPZ_DIR = ROOT / "data/processed/npz_96x96x96"
METADATA_CSV = ROOT / "data/splits/splits_metadata.csv"
OUTPUT_DIR = ROOT / "outputs/tables/tv4_week2"

RUNS = [
    {
        "run_id": "M2_IP3_C1",
        "checkpoint": ROOT / "outputs/checkpoints/M2_IP3_C1/best.pt",
        "arch": "M2",
        "ip_mode": "IP3_multi",
        "use_film": True,
        "state_key": "model",
        "model_kwargs": {},
    },
]
MAX_CASES = 1

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_BF16 = DEVICE.type == "cuda" and torch.cuda.is_bf16_supported()
ROI_SIZE = (96, 96, 96)
SW_BATCH_SIZE = 1
SW_OVERLAP = 0.5

SPACING_DHW_MM = (1.0, 1.0, 1.0)
PROB_THRESHOLD = 0.5
MIN_COMPONENT_VOXELS = 50
SECONDARY_VOLUME_RATIO = 0.10
SECONDARY_MAX_DISTANCE_MM = 30.0
CLOSING_RADIUS_MM = 2.0

## 2. Dữ liệu test và bàn giao checkpoint

**TV3 · checkpoint M2.** Điền đường dẫn file `.pt` và cấu hình đã dùng khi train vào `RUNS`. `ip_mode` nhận `IP1_post1`, `IP2_sub` hoặc `IP3_multi`. Chọn `use_film` theo run. Nếu bề rộng backbone hoặc FiLM khác mặc định, truyền các tham số tương ứng của `build_model` qua `model_kwargs`.

`state_key` mặc định là `"model"`, theo định dạng trainer của repo. Đổi thành key thực tế, ví dụ `"model_state_dict"`, hoặc đặt `None` nếu file lưu trực tiếp state dict. Nếu checkpoint có cấu trúc lồng khác, chỉnh `read_state_dict` bên dưới. Cùng cấu hình này dùng được cho M1 từ TV2 bằng cách đổi `arch` và các thông số của run.

Bộ đọc dữ liệu lấy toàn bộ VOI từ split test, giữ nguyên kích thước và thứ tự trục D/H/W. Khoảng cách dùng spacing sau resample của Steps 1–3, mặc định 1 mm theo cả ba trục.

In [ ]:
def make_test_dataset(ip_mode):
    return MAMA_MIA_Dataset(
        metadata_csv=METADATA_CSV,
        npz_dir=NPZ_DIR,
        split="test",
        ip_mode=ip_mode,
        augment=False,
        cache_rate=0.0,
    )


def read_state_dict(checkpoint_path, state_key):
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    if state_key is not None and state_key not in checkpoint:
        raise KeyError(f"{checkpoint_path.name}: không có key {state_key!r}. Kiểm tra state_key trong RUNS.")

    state_dict = checkpoint if state_key is None else checkpoint[state_key]
    if not state_dict or not all(torch.is_tensor(value) for value in state_dict.values()):
        raise ValueError(f"{checkpoint_path.name}: state_key phải trỏ đến state dict của mô hình.")

    while True:
        prefix = next(
            (p for p in ("_orig_mod.", "module.") if all(key.startswith(p) for key in state_dict)),
            None,
        )
        if prefix is None:
            return state_dict
        torch.nn.modules.utils.consume_prefix_in_state_dict_if_present(state_dict, prefix)


def load_model(run):
    model = build_model(
        arch=run["arch"],
        ip_mode=run["ip_mode"],
        use_film=run["use_film"],
        **run["model_kwargs"],
    )
    weights = read_state_dict(run["checkpoint"], run["state_key"])
    model.load_state_dict(weights, strict=True)
    return model.to(DEVICE).eval()

## 3. Sliding Window và 8-Way Flip TTA

Mỗi lượt dự đoán ghép logits từ các cửa sổ bằng trọng số Gaussian với overlap 50%. Lật ngược xác suất về hệ trục ban đầu rồi lấy trung bình qua 8 tổ hợp lật, gồm cả lượt giữ nguyên. Với FiLM, toàn bộ cửa sổ của một ca nhận cùng subtype.

Mỗi lượt TTA chạy tuần tự. MONAI chuyển từng batch cửa sổ sang thiết bị suy luận và ghép kết quả trên CPU. `SW_BATCH_SIZE = 1` là cấu hình khởi đầu.

In [ ]:
FLIP_AXES = tuple(
    tuple(axis for axis, flip in zip((2, 3, 4), flags) if flip)
    for flags in product((False, True), repeat=3)
)


@torch.inference_mode()
def predict_probability(model, sample):
    image = sample["image"].unsqueeze(0)
    subtype = sample["subtype_id"].reshape(1).to(DEVICE)

    def predict_windows(windows):
        with torch.autocast(device_type=DEVICE.type, dtype=torch.bfloat16, enabled=USE_BF16):
            logits = model(windows, subtype=subtype.expand(windows.shape[0]))
        return logits.float()

    probability_sum = torch.zeros((1, 1, *image.shape[2:]), dtype=torch.float32)
    for axes in FLIP_AXES:
        oriented = torch.flip(image, axes) if axes else image
        logits = sliding_window_inference(
            inputs=oriented,
            roi_size=ROI_SIZE,
            sw_batch_size=SW_BATCH_SIZE,
            predictor=predict_windows,
            overlap=SW_OVERLAP,
            mode="gaussian",
            sw_device=DEVICE,
            device=torch.device("cpu"),
        )
        probability = logits.sigmoid()
        probability_sum += torch.flip(probability, axes) if axes else probability

    probability = (probability_sum / len(FLIP_AXES))[0, 0].numpy()
    if not np.isfinite(probability).all():
        raise ValueError(f"{sample['patient_id']}: xác suất dự đoán chứa NaN hoặc Inf.")
    return probability

## 4. Mask P0/P1 và chỉ số

- **P0** · Ngưỡng xác suất 0,5.
- **P1** · Từ P0, lọc thành phần liên thông 26 lân cận. Giữ thành phần lớn nhất nếu đạt 50 voxel. Cụm phụ phải đạt 50 voxel, lớn hơn 10% thể tích cụm chính và có trọng tâm cách cụm chính tối đa 30 mm. Sau đó đóng hình thái học bằng phần tử cấu trúc cầu bán kính 2 mm, rồi lấp lỗ trong khối 3D và trên ba mặt phẳng.
- **Điểm** · DSC, Recall, Precision tính theo phần trăm. HD95 đối xứng tính bằng mm. Hai mask rỗng được tính là khớp hoàn toàn. Nếu chỉ một mask rỗng, DSC, Recall và Precision bằng 0, còn HD95 để trống. Bảng tổng hợp ghi số ca có HD95 hợp lệ.

In [ ]:
def filter_components(mask, spacing):
    labels, _ = ndimage.label(mask, structure=ndimage.generate_binary_structure(3, 3))
    sizes = np.bincount(labels.ravel())
    sizes[0] = 0
    primary = int(sizes.argmax())
    if sizes[primary] < MIN_COMPONENT_VOXELS:
        return np.zeros_like(mask)

    secondary = np.flatnonzero(
        (sizes >= MIN_COMPONENT_VOXELS) & (sizes > SECONDARY_VOLUME_RATIO * sizes[primary])
    )
    secondary = secondary[secondary != primary]
    keep = [primary]
    if secondary.size:
        centers = np.asarray(ndimage.center_of_mass(mask, labels, [primary, *secondary]))
        distance_mm = np.linalg.norm((centers[1:] - centers[0]) * spacing, axis=1)
        keep.extend(secondary[distance_mm <= SECONDARY_MAX_DISTANCE_MM])
    return np.isin(labels, keep)


def postprocess(mask, spacing=SPACING_DHW_MM):
    spacing = np.asarray(spacing, dtype=float)
    mask = filter_components(mask, spacing)
    if not mask.any():
        return mask

    radii = np.ceil(CLOSING_RADIUS_MM / spacing).astype(int)
    grid = np.ogrid[tuple(slice(-r, r + 1) for r in radii)]
    distance_sq = sum((coords * step) ** 2 for coords, step in zip(grid, spacing))
    structure = distance_sq <= CLOSING_RADIUS_MM ** 2
    padded = np.pad(mask, [(r, r) for r in radii], mode="constant")
    closed = ndimage.binary_closing(padded, structure=structure)
    crop = tuple(slice(r, r + size) for r, size in zip(radii, mask.shape))

    filled_3d = ndimage.binary_fill_holes(closed[crop])
    filled = filled_3d.copy()
    for axis in range(3):
        source = np.moveaxis(filled_3d, axis, 0)
        target = np.moveaxis(filled, axis, 0)
        for index, plane in enumerate(source):
            target[index] |= ndimage.binary_fill_holes(plane)
    return filled


def score_masks(prediction, target, spacing=SPACING_DHW_MM):
    if prediction.shape != target.shape:
        raise ValueError(f"Mask dự đoán {prediction.shape} khác kích thước nhãn {target.shape}.")

    pred_count = np.count_nonzero(prediction)
    target_count = np.count_nonzero(target)
    if not pred_count or not target_count:
        both_empty = not (pred_count or target_count)
        score = 100.0 if both_empty else 0.0
        return {
            "dsc_pct": score,
            "hd95_mm": 0.0 if both_empty else np.nan,
            "recall_pct": score,
            "precision_pct": score,
        }

    intersection = np.count_nonzero(prediction & target)
    hd95 = compute_hausdorff_distance(
        torch.from_numpy(prediction)[None, None],
        torch.from_numpy(target)[None, None],
        include_background=True,
        percentile=95,
        spacing=spacing,
    ).item()
    return {
        "dsc_pct": 200.0 * intersection / (pred_count + target_count),
        "hd95_mm": hd95,
        "recall_pct": 100.0 * intersection / target_count,
        "precision_pct": 100.0 * intersection / pred_count,
    }

## 5. Chạy đánh giá

Một dự đoán cho mỗi ca và checkpoint được dùng để chấm cả hai chế độ. Sau lượt một ca, đối chiếu `patient_id`, kích thước mask và hai dòng P0/P1 trong bảng chi tiết trước khi chạy toàn bộ split.

In [ ]:
def evaluate_run(run):
    model = load_model(run)
    dataset = make_test_dataset(run["ip_mode"])
    case_count = len(dataset) if MAX_CASES is None else min(MAX_CASES, len(dataset))
    rows = []
    print(f"{run['run_id']} · {case_count}/{len(dataset)} ca · {DEVICE}")

    for index in range(case_count):
        sample = dataset[index]
        target = sample["mask"][0].numpy().astype(bool)
        p0 = predict_probability(model, sample) >= PROB_THRESHOLD
        p1 = postprocess(p0)
        for mode, prediction in (("P0", p0), ("P1", p1)):
            rows.append({
                "run_id": run["run_id"],
                "arch": run["arch"],
                "ip_mode": run["ip_mode"],
                "use_film": run["use_film"],
                "patient_id": sample["patient_id"],
                "dataset": sample["dataset"],
                "tumor_subtype": sample["tumor_subtype"],
                "mode": mode,
                **score_masks(prediction, target),
            })
        if (index + 1) % 20 == 0 or index + 1 == case_count:
            print(f"  {index + 1}/{case_count}")
    return rows

In [ ]:
if not RUNS or len({run["run_id"] for run in RUNS}) != len(RUNS):
    raise ValueError("RUNS phải có ít nhất một cấu hình và các run_id phải khác nhau.")
if MAX_CASES is not None and MAX_CASES < 1:
    raise ValueError("MAX_CASES phải là số nguyên dương hoặc None.")

missing = [run["checkpoint"] for run in RUNS if not run["checkpoint"].is_file()]
if missing:
    raise FileNotFoundError("Điền đường dẫn checkpoint đã bàn giao trong RUNS:\n" + "\n".join(map(str, missing)))

per_case = pd.DataFrame(row for run in RUNS for row in evaluate_run(run))
display(per_case.head())

## 6. Xuất kết quả

Bảng tổng hợp lấy trung bình theo ca cho từng cấu hình và chế độ. `hd95_valid` là số ca được dùng để tính trung bình HD95. Các lượt giới hạn số ca nằm trong thư mục `preview`, các lượt chạy toàn bộ tập test nằm trong `full_test`.

In [ ]:
def export_results(per_case, output_dir):
    summary = (
        per_case.groupby(["run_id", "arch", "ip_mode", "use_film", "mode"], sort=False)
        .agg(
            cases=("patient_id", "size"),
            hd95_valid=("hd95_mm", "count"),
            dsc_pct=("dsc_pct", "mean"),
            hd95_mm=("hd95_mm", "mean"),
            recall_pct=("recall_pct", "mean"),
            precision_pct=("precision_pct", "mean"),
        )
        .reset_index()
    )

    output_dir.mkdir(parents=True, exist_ok=True)
    per_case.to_csv(output_dir / "per_case.csv", index=False)
    summary.to_csv(output_dir / "summary.csv", index=False)

    table = summary.round(4).astype("string").fillna("—")
    lines = [
        "| " + " | ".join(table.columns) + " |",
        "| " + " | ".join(["---"] * len(table.columns)) + " |",
    ]
    lines.extend("| " + " | ".join(row) + " |" for row in table.itertuples(index=False, name=None))
    (output_dir / "summary.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
    return summary

In [ ]:
result_dir = OUTPUT_DIR / ("full_test" if MAX_CASES is None else "preview")
summary = export_results(per_case, result_dir)
display(summary)
print(result_dir)